# Initial Settings

In [1]:
import pandas as pd
import numpy as np

%config Completer.use_jedi = False

# Synoptic Data

In [2]:
df = pd.read_csv(filepath_or_buffer="Data/Stations_Data.csv")

df.Date_Miladi = pd.to_datetime(arg=df.Date_Miladi,
                                format='%Y-%m-%d')

Stations = df.groupby([df['Station'].rename("Station"),
                       df['Date_Miladi'].dt.year.rename("Year"),
                       df['Date_Miladi'].dt.month.rename("Month")])['Precip'].sum().reset_index()

Stations = Stations.drop(Stations.query('(Year == 2001 and Month <= 9)').index).reset_index(drop=True)
Stations = Stations.drop(Stations.query('(Year >= 2016)').index).reset_index(drop=True)
Stations = Stations.drop(Stations.query('(Year == 2015 and Month >= 9)').index).reset_index(drop=True)
Stations.sort_values(by=["Station", "Year", "Month"], inplace=True)
Stations

,Station,Year,Month,Precip
0,ABADAN,2001,10,2.40
1,ABADAN,2001,11,6.90
2,ABADAN,2001,12,123.42
3,ABADAN,2002,1,29.20
4,ABADAN,2002,2,22.61
...,...,...,...,...
7844,TANG PANJ - BAKHTIYARI,2015,4,91.00
7845,TANG PANJ - BAKHTIYARI,2015,5,32.00
7846,TANG PANJ - BAKHTIYARI,2015,6,0.00
7847,TANG PANJ - BAKHTIYARI,2015,7,0.00


# Load geeID

In [3]:
geeID = pd.read_csv("Data/geeID.csv")

geeID

,geeID,TYPE,Station,LAT,LONG,ELEV,ELEV_SRTM3
0,1,SYN,ABADAN,30.377245,48.214718,6.6,2
1,2,EVAP,ABDOLLAHKHAN,31.833333,48.383333,40.0,39
2,3,SYN,AHVAZ,31.333333,48.683333,22.5,14
3,4,EVAP,AHVAZ - EVAP,31.344766,48.744172,10.0,22
4,5,SYN,ALIGUDARZ,33.408076,49.702501,2022.1,2022
5,6,EVAP,ARAB HASAN - ASAD,31.850000,48.883333,33.0,27
6,7,EVAP,BAGH MALEK,31.524444,49.866667,675.0,643
7,8,SYN,BANDAR MAHSHAHR,30.545539,49.159175,6.2,5
8,9,EVAP,BARANGERD,31.716667,49.816667,908.0,908
9,10,SYN,BEHBAHAN,30.605557,50.217219,313.0,308


# TerraClimate Data

In [4]:
TerraClimate = pd.read_csv("Data/TerraClimate.csv")
TerraClimate["Date"] = TerraClimate.startDate.str.split("T", 1, expand=True)[0]
TerraClimate["Date"] = pd.to_datetime(arg=TerraClimate["Date"], format='%Y-%m-%d')
TerraClimate["Year"] = TerraClimate["Date"].dt.year
TerraClimate["Month"] = TerraClimate["Date"].dt.month
TerraClimate["TerraClimate"] = TerraClimate["mean"]
TerraClimate = TerraClimate[['geeID', 'Year', 'Month', 'TerraClimate']]
TerraClimate = TerraClimate.drop(TerraClimate.query('(Year == 2001 and Month == [1, 2, 3, 4, 5, 6, 7, 8, 9])').index).reset_index(drop=True)
TerraClimate = TerraClimate.drop(TerraClimate.query('(Year == 2015 and Month == [9, 10, 11, 12])').index).reset_index(drop=True)

TerraClimate = pd.merge(left=TerraClimate,
                        right=geeID,
                        how='left',
                        on=['geeID'])[['geeID', 'Station', 'TYPE', 'LAT', 'LONG', 'ELEV', 'ELEV_SRTM3', 'Year', 'Month', 'TerraClimate']]

TerraClimate.sort_values(by=["Station", "Year", "Month"], inplace=True)

TerraClimate

,geeID,Station,TYPE,LAT,LONG,ELEV,ELEV_SRTM3,Year,Month,TerraClimate
2,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,10,3.0
49,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,11,8.0
96,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,12,104.0
143,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2002,1,27.0
190,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2002,2,20.0
...,...,...,...,...,...,...,...,...,...,...
7660,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,4,13.0
7707,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,5,17.0
7754,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,6,1.0
7801,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,7,0.0


# ERA5 Monthly Aggregate

In [5]:
ERA5_Monthly_Aggregate = pd.read_csv("Data/ERA5_Monthly_Aggregate.csv")
ERA5_Monthly_Aggregate["Date"] = ERA5_Monthly_Aggregate.startDate.str.split("T", 1, expand=True)[0]
ERA5_Monthly_Aggregate["Date"] = pd.to_datetime(arg=ERA5_Monthly_Aggregate["Date"], format='%Y-%m-%d')
ERA5_Monthly_Aggregate["Year"] = ERA5_Monthly_Aggregate["Date"].dt.year
ERA5_Monthly_Aggregate["Month"] = ERA5_Monthly_Aggregate["Date"].dt.month
ERA5_Monthly_Aggregate["ERA5_Monthly_Aggregate"] = ERA5_Monthly_Aggregate["mean"]
ERA5_Monthly_Aggregate = ERA5_Monthly_Aggregate[['geeID', 'Year', 'Month', 'ERA5_Monthly_Aggregate']]
ERA5_Monthly_Aggregate = ERA5_Monthly_Aggregate.drop(ERA5_Monthly_Aggregate.query('(Year == 2001 and Month == [1, 2, 3, 4, 5, 6, 7, 8, 9])').index).reset_index(drop=True)
ERA5_Monthly_Aggregate = ERA5_Monthly_Aggregate.drop(ERA5_Monthly_Aggregate.query('(Year == 2015 and Month == [9, 10, 11, 12])').index).reset_index(drop=True)

ERA5_Monthly_Aggregate = pd.merge(left=ERA5_Monthly_Aggregate,
                                  right=geeID,
                                  how='left',
                                  on=['geeID'])[['geeID', 'Station', 'TYPE', 'LAT', 'LONG', 'ELEV', 'ELEV_SRTM3', 'Year', 'Month', 'ERA5_Monthly_Aggregate']]

ERA5_Monthly_Aggregate["ERA5_Monthly_Aggregate"] = ERA5_Monthly_Aggregate["ERA5_Monthly_Aggregate"] * 1000

ERA5_Monthly_Aggregate.sort_values(by=["Station", "Year", "Month"], inplace=True)

ERA5_Monthly_Aggregate

,geeID,Station,TYPE,LAT,LONG,ELEV,ELEV_SRTM3,Year,Month,ERA5_Monthly_Aggregate
2,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,10,0.959212
49,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,11,2.042739
96,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,12,130.619898
143,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2002,1,34.397982
190,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2002,2,18.392704
...,...,...,...,...,...,...,...,...,...,...
7660,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,4,54.141205
7707,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,5,18.780990
7754,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,6,0.224290
7801,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,7,2.615742


# Combine Data

In [6]:
data = pd.merge(left=Stations,
                right=TerraClimate, 
                how='left',
                on=['Station', 'Year', 'Month'])

data = pd.merge(left=data,
                right=ERA5_Monthly_Aggregate[['Station', 'Year', 'Month', 'ERA5_Monthly_Aggregate']], 
                how='left',
                on=['Station', 'Year', 'Month'])

data = data[["geeID", "Station", "TYPE", "LAT", "LONG", "ELEV", "ELEV_SRTM3", "Year", "Month", "Precip", "TerraClimate", "ERA5_Monthly_Aggregate"]]

data

,geeID,Station,TYPE,LAT,LONG,ELEV,ELEV_SRTM3,Year,Month,Precip,TerraClimate,ERA5_Monthly_Aggregate
0,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,10,2.40,3.0,0.959212
1,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,11,6.90,8.0,2.042739
2,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2001,12,123.42,104.0,130.619898
3,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2002,1,29.20,27.0,34.397982
4,1,ABADAN,SYN,30.377245,48.214718,6.6,2,2002,2,22.61,20.0,18.392704
...,...,...,...,...,...,...,...,...,...,...,...,...
7844,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,4,91.00,13.0,54.141205
7845,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,5,32.00,17.0,18.780990
7846,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,6,0.00,1.0,0.224290
7847,47,TANG PANJ - BAKHTIYARI,EVAP,32.933333,48.766667,645.0,645,2015,7,0.00,0.0,2.615742


# Save Data

In [7]:
data["ERA5_Error"] = data["Precip"] - data["ERA5_Monthly_Aggregate"]

data.to_csv(path_or_buf="Output/data_monthly.csv", mode="w", index=False)

data_yearly = data.groupby(['Station', 'Year']).agg({'Precip': 'sum',
                                                     'TerraClimate': 'sum',
                                                     'ERA5_Monthly_Aggregate': 'sum',
                                                     'ERA5_Error': 'sum'}).reset_index()

data_yearly = pd.merge(left=geeID,
                       right=data_yearly,
                       how='left',
                       on=['Station'])

data_yearly.to_csv(path_or_buf="Output/data_yearly.csv", mode="w", index=False)

# Create Error Raster

In [9]:
for year in data.Year.unique():
    for month in data.Month.unique():
        data.query('Year == @year').query('Month == @month').to_csv("Output/YearMonth/Year_" + str(year) + "_Month_" + str(month) + ".csv", index=False)